# 🎯 [ISR03] 드론 비행로그 패턴 이상 탐지와 운용 주체 단정 방지
### Google Colab 실습 환경 — 미션 04 (isr03_m04_solution.ipynb) (정보·감시·정찰(ISR) / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/ISR03/isr03_m04_solution.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 정보·감시·정찰(ISR) (Level 3) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.5시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 상용 드론이 정찰과 공격에 폭넓게 쓰이면서, 노획하거나 수집한 드론 비행로그를 분석해 이상 비행을 찾아내는 일이 중요해졌다. 그러나 같은 상용 기체를 아군·민간·적대 세력이 모두 쓰기 때문에, 비행 특성만 보고 운용 주체를 성급하게 단정하면 오판으로 이어질 수 있다. 본 프로젝트에서는 교육용 가상 비행로그를 코드로 직접 만들어 스키마 정의, GPS 스푸핑 보정 전처리, 규칙 기반 Baseline, 군집·비지도 학습 기반 이상 탐지, 불확실성을 반영한 운용 주체 신뢰도 엔진, 분석 대시보드, 정보분석관 심층 검토(HITL), 대량 로그 부하 시험, AAR 재보정, 최종 통합까지 단계별로 구현한다. 최종 목표는 이상 비행을 놓치지 않으면서 오탐을 줄이고, 운용 주체에 관한 판단은 항상 확률·불확실성·근거와 함께 사람의 검토를 거쳐 보고되도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 🛠️ 2. 실습 실행 및 산출물 저장·데이터 연계 가이드

1. **Colab / 로컬 실행 환경 안내**
   - 아래 환경 설정 셀의 `!pip install -q ...` 구문은 **Google Colab / Jupyter 노트북 전용** 매직 커맨드입니다.
   - 일반 로컬 Python(`.py`) 환경에서 실행할 때는 해당 줄을 제거하고 터미널에서 `pip install numpy pandas matplotlib scipy scikit-learn`을 먼저 실행하세요.

2. **산출물(`deliverables`) 디렉토리 구조 및 파일 저장 방법**
   - 본 노트북의 각 미션 셀은 Colab 환경에서 단독 실행이 가능하도록 인메모리 시뮬레이션으로 구성되어 있습니다.
   - 프로젝트 제출용 산출물(`data/schema/`, `docs/`, `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `tests/`, `configs/`)을 실제 파일로 저장하려면, 환경 설정 셀에 제공된 `init_deliverable_dirs()` 및 `save_deliverable(rel_path, content)` 헬퍼 함수 또는 셀 상단의 `%%writefile <파일경로>` 매직 커맨드를 사용하세요.

3. **첨부 데이터셋(`isr03_mXX_dataset.json`) 연계 활용**
   - 각 미션 셀은 외부 파일 의존성 없이도 독립 실행되도록 자체 난수 기반 가상 데이터를 생성합니다.
   - 미션별 첨부파일(`isr03_m01_dataset.json` ~ `isr03_m10_dataset.json`, `isr03_mXX_sample.md`)은 `load_mission_dataset(1)` 함수로 불러와 스키마 대조, 경계값 추가 검증, 파라미터 확장 실험에 연계 활용할 수 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# 산출물 디렉토리 초기화 및 파일 저장/첨부 데이터셋 로드 헬퍼
DELIVERABLE_DIRS = [
    'data/schema', 'docs', 'src/preprocess', 'src/models',
    'src/verification', 'src/ui', 'src/hitl', 'tests', 'configs'
]

def init_deliverable_dirs(base_dir='.'):
    """프로젝트 필수 산출물 디렉토리를 생성한다."""
    for d in DELIVERABLE_DIRS:
        os.makedirs(os.path.join(base_dir, d), exist_ok=True)

def save_deliverable(rel_path, content, base_dir='.'):
    """산출물 파일(JSON, YAML, MD, PY 등)을 지정 경로에 저장한다."""
    full_path = os.path.join(base_dir, rel_path)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, 'w', encoding='utf-8') as f:
        f.write(content if isinstance(content, str) else json.dumps(content, ensure_ascii=False, indent=2))
    return full_path

def load_mission_dataset(mission_num, base_dir='.'):
    """첨부된 미션별 가상 데이터셋(isr03_mXX_dataset.json)을 불러온다."""
    fname = f'isr03_m{int(mission_num):02d}_dataset.json'
    for candidate in [os.path.join(base_dir, fname), os.path.join(base_dir, 'uploads', 'ISR03', fname)]:
        if os.path.exists(candidate):
            with open(candidate, encoding='utf-8') as f:
                return json.load(f)
    raise FileNotFoundError(f'{fname} 파일을 찾을 수 없습니다. Colab에 업로드했는지 확인하세요.')


---

## 🎯 미션 04. 비행 패턴 클러스터링 및 머신러닝 기반 은밀 침투 드론 이상 궤적 탐지 모델 고도화
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

비행 패턴 클러스터링 및 비지도 머신러닝을 적용하여 은밀 침투 드론의 특이 비행 로그 및 비인가 비행 구역 진입 패턴을 정밀 탐지한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **평시 비행 패턴 군집화와 비지도 이상 탐지를 적용해 규칙으로 잡기 어려운 은밀 침투·저고도·조종신호 무응답·선회 체류 등 복합 이상 비행을 탐지하도록 고도화한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **30초 창 단위로 고도·속도·선회·직진성·조종신호·비인가 구역 접근·경로 이탈 특징을 계산하고, 라벨 없는 평시 로그로 KMeans 통상 패턴 군집과 IsolationForest를 학습 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/models/isr03_advanced_detector.py 및 예외 탐지 테스트 결과` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 기만/교란 공격에 대한 방어 로직의 현실성 및 신뢰성

#### 미션 PASS 조건

- 정량 기준: 복합 위협 탐지 재현율(Recall) 85% 이상, 처리 지연 200ms 이하


- 진행 기준: 복합 위협 시나리오 10종에 대한 탐지 테스트 통과


- 검증 방법: 합성 위협 데이터셋 주입 시 이상치 탐지 정확도 검증


- AI 검증 기준: AI의 특징 재설계·z-score 해석을 실제 시나리오 10종 실험 수치와 대조 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""평시 비행로그로 통상 비행 패턴을 군집화(KMeans)하고 비지도 이상 탐지(IsolationForest)를 결합해, 규칙으로 잡기 어려운 저고도·조종신호 무응답·선회 체류·은밀 침투 등 10종 복합 이상 비행을 탐지하고 재현율·오경보·처리 지연을 측정한다."""
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------------
# [교육용가상자료] 합성 드론 비행로그 생성기 (실제 좌표·기체·부대 정보 없음)
# 가상 구역(원점 기준 동·북 방향 m 좌표)에서 1초 간격 비행로그와 정답(이상 구간·운용 주체)을 만든다.
# ---------------------------------------------------------------------------
NFZ = {"x": 2500.0, "y": 2500.0, "r": 700.0}        # 가상 비인가 비행구역
WATCH = {"x": -2000.0, "y": 1500.0, "r": 600.0}     # 가상 관심 구역
SCENARIOS = ["sharp_turn", "hover", "dive", "deviation", "nfz_entry",
             "low_level", "rc_silent", "infiltration", "loiter", "speed_burst"]
OPERATORS = ["friendly", "civil", "hostile"]
# 시나리오별 운용 주체 분포: 일부러 크게 겹치게 설계했다(비행 특성만으로 운용 주체를 단정할 수 없다).
OPERATOR_P = {"normal": (.45, .47, .08), "survey": (.15, .82, .03), "sharp_turn": (.40, .35, .25),
              "hover": (.25, .45, .30), "dive": (.25, .25, .50), "deviation": (.30, .40, .30),
              "nfz_entry": (.10, .50, .40), "low_level": (.35, .15, .50), "rc_silent": (.30, .15, .55),
              "infiltration": (.12, .08, .80), "loiter": (.30, .20, .50), "speed_burst": (.30, .20, .50)}
PLATFORM_P = {"friendly": (.40, .55, .05), "civil": (.88, .02, .10), "hostile": (.55, .30, .15)}   # commercial/military/unknown
FIRMWARE_P = {"friendly": (.30, .65, .05), "civil": (.85, .10, .05), "hostile": (.20, .50, .30)}   # stock/custom/unknown


def _wrap(a):
    return (a + 180.0) % 360.0 - 180.0


def _bearing(p, q):
    return math.degrees(math.atan2(q[0] - p[0], q[1] - p[1])) % 360.0


def _away_from_nfz(rng, min_d):
    while True:
        c = rng.uniform(-3800, 3800, 2)
        if np.hypot(c[0] - NFZ["x"], c[1] - NFZ["y"]) > min_d:
            return c


def make_flight(fid, kind, rng, n=300, weather=None):
    """비행 1회 = (로그 DataFrame, 메타 dict). kind: normal / survey / SCENARIOS 중 하나"""
    weather = weather or str(rng.choice(["clear", "night", "windy", "rain"], p=[.4, .25, .2, .15]))
    wind = {"clear": 1.0, "night": 1.0, "windy": 2.5, "rain": 1.8}[weather]
    v0, alt0 = rng.uniform(11, 17), rng.uniform(70, 150)
    nfz_c = np.array([NFZ["x"], NFZ["y"]])
    if kind == "infiltration":                      # 원거리에서 비인가 구역 방향으로 저고도 직진
        ang = rng.uniform(0, 2 * np.pi)
        start = nfz_c + 4000 * np.array([np.cos(ang), np.sin(ang)])
        plan = np.array([start, nfz_c])
        v0, alt0 = rng.uniform(9, 11), rng.uniform(25, 35)
    else:
        if kind == "nfz_entry":                     # 비인가 구역 1.6km 밖에서 순찰하다가 진입
            ang = rng.uniform(0, 2 * np.pi)
            c = nfz_c + 1600 * np.array([np.cos(ang), np.sin(ang)])
        else:
            c = _away_from_nfz(rng, 2600)
        if kind == "survey":                        # 민간 측량형 왕복(지그재그) 경로
            w, h = rng.uniform(500, 700), 120.0
            plan = np.array([c + [(-w if i % 4 in (0, 3) else w), i // 2 * h - 360] for i in range(8)])
        else:                                       # 사각 순찰 경로(웨이포인트 4개)
            a, b, th = rng.uniform(300, 600), rng.uniform(200, 400), rng.uniform(0, np.pi)
            R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
            plan = c + np.array([[-a, -b], [a, -b], [a, b], [-a, b]]) @ R.T
    s0 = int(rng.integers(80, 160))
    span = {"sharp_turn": 30, "hover": 60, "dive": 8, "deviation": 70, "nfz_entry": 120, "low_level": 120,
            "rc_silent": 120, "infiltration": n - 20, "loiter": 90, "speed_burst": 25}.get(kind, 0)
    if kind == "infiltration":
        s0 = 20
    win = (s0, s0 + span)
    turns = set(rng.choice(np.arange(s0, s0 + 30), 3, replace=False).tolist()) if kind == "sharp_turn" else set()
    # 난수를 미리 한 번에 뽑아 두고(속도 향상) 1초 단위로 기체 운동을 적분한다.
    e_h, e_v, e_a = rng.normal(0, 0.8 * wind, n), rng.normal(0, 0.3 * wind, n), rng.normal(0, 0.4 * wind, n)
    rc_arr = np.clip(rng.normal(rng.uniform(60, 92), 6, n), 0, 100)
    gust = (rng.random(n) < (0.004 if weather == "windy" else 0.0)) * rng.uniform(20, 35, n)
    burst, turn_amt = rng.uniform(23, 28), rng.choice([-1, 1], n) * rng.uniform(100, 160, n)
    side = float(rng.choice([-90, 90]))
    low_alt, rng_rc = rng.uniform(35, 55), rng.uniform(0, 20, n)          # 저고도 목표, 조종신호 두절 시 잔여 수신값
    x, y, wi = float(plan[0][0]), float(plan[0][1]), 1
    hdg, alt = _bearing((x, y), plan[1]), alt0
    max_rate = 40.0 if kind == "survey" else 12.0
    dev_target, X, Y, A, V, HD = None, np.zeros(n), np.zeros(n), np.zeros(n), np.zeros(n), np.zeros(n)
    for t in range(n):
        inw = win[0] <= t < win[1]
        tx, ty = plan[1] if kind == "infiltration" else plan[wi % len(plan)]
        spd_cmd, alt_cmd = v0, alt0
        if inw:
            if kind == "deviation":
                if dev_target is None:
                    r = math.radians(hdg + side)
                    dev_target = (x + 900 * math.sin(r), y + 900 * math.cos(r))
                tx, ty = dev_target
            elif kind == "nfz_entry":
                tx, ty = NFZ["x"], NFZ["y"]
            elif kind == "hover":
                spd_cmd = 0.0
            elif kind == "speed_burst":
                spd_cmd = burst
            elif kind == "low_level":
                alt_cmd = low_alt
            if kind in ("rc_silent", "infiltration"):
                rc_arr[t] = rng_rc[t]
        if inw and kind == "loiter":
            hdg = (hdg + 9.0 + e_h[t] / wind) % 360
        else:
            d = _wrap(math.degrees(math.atan2(tx - x, ty - y)) - hdg)
            hdg = (hdg + max(-max_rate, min(max_rate, d)) + e_h[t]) % 360
        if t in turns:
            hdg = (hdg + turn_amt[t]) % 360
        v = abs(spd_cmd + e_v[t]) if spd_cmd > 0 else abs(e_v[t] / wind)
        x += v * math.sin(math.radians(hdg)); y += v * math.cos(math.radians(hdg))
        if kind != "infiltration" and math.hypot(x - tx, y - ty) < 40 and not (inw and kind in ("deviation", "nfz_entry")):
            wi += 1
        if inw and kind == "dive":
            alt -= 9.0 + e_v[t]
        else:
            alt += max(-3.0, min(3.0, alt_cmd - alt)) + e_a[t]
        alt -= gust[t]                                     # 돌풍 하강기류(정상 비행에서도 발생)
        if kind == "dive" and t >= win[1]:
            alt0 = min(alt0, alt)
        alt = max(alt, 3.0)
        X[t], Y[t], A[t], V[t], HD[t] = x, y, alt, v, hdg
    gps_sd = {"clear": 2.0, "night": 2.0, "windy": 3.0, "rain": 4.5}[weather]     # 악천후일수록 GPS 잡음(다중경로) 증가
    df = pd.DataFrame({"t": np.arange(n), "x": X + rng.normal(0, gps_sd, n), "y": Y + rng.normal(0, gps_sd, n),
                       "alt": A + rng.normal(0, 0.5, n), "speed": V + rng.normal(0, 0.2, n),
                       "heading": (HD + rng.normal(0, 1.5, n)) % 360, "rc": rc_arr.round(1),
                       "sats": rng.integers(10, 15, n), "hdop": rng.uniform(0.6, 1.2, n).round(2),
                       "x_true": X, "y_true": Y})
    df["anomaly"] = (df["t"] >= win[0]) & (df["t"] < win[1]) & (kind in SCENARIOS)
    op = OPERATORS[int(rng.choice(3, p=OPERATOR_P[kind]))]
    meta = {"id": fid, "kind": kind, "weather": weather, "operator": op, "plan": plan, "window": win if kind in SCENARIOS else None,
            "platform": ["commercial", "military", "unknown"][int(rng.choice(3, p=PLATFORM_P[op]))],
            "firmware": ["stock", "custom", "unknown"][int(rng.choice(3, p=FIRMWARE_P[op]))]}
    return df, meta


def corrupt(df, rng, spoof=True, emi=True, drop=True, dup=True):
    """전장 잡음 주입: GPS 스푸핑(위치 점프·표류), 전자기 간섭(고도 스파이크), 센서 단절(결측), 중복 기록."""
    d = df.copy()
    d["spoof_true"] = False
    if spoof:
        a = int(rng.integers(40, len(d) - 60)); L = int(rng.integers(25, 45))
        th = rng.uniform(0, 2 * np.pi)
        drift = rng.uniform(22, 28) * np.arange(L)[:, None] * np.array([np.cos(th), np.sin(th)])
        jump = rng.uniform(250, 400) * np.array([np.cos(th + 1), np.sin(th + 1)])
        d.loc[a:a + L - 1, ["x", "y"]] += drift + jump
        d.loc[a:a + L - 1, "spoof_true"] = True
    if emi:
        idx = rng.choice(len(d), 6, replace=False)
        d.loc[idx, "alt"] += rng.choice([-1, 1], 6) * rng.uniform(60, 150, 6)
    if drop:
        a = int(rng.integers(10, len(d) - 20))
        d.loc[a:a + int(rng.integers(4, 9)), ["x", "y", "alt", "speed", "heading"]] = np.nan
        d = d.drop(index=rng.choice(len(d), 5, replace=False))
    if dup:
        d = pd.concat([d, d.sample(5, random_state=int(rng.integers(1e6)))]).sort_values("t", kind="stable")
    return d.reset_index(drop=True)


def make_dataset(n=200, seed=7, anomaly_ratio=0.5, scenarios=SCENARIOS):
    rng = np.random.default_rng(seed)
    out = []
    for i in range(n):
        if i % 10 < anomaly_ratio * 10:
            kind = scenarios[(i // 10 * int(anomaly_ratio * 10) + i % 10) % len(scenarios)]
        else:
            kind = "survey" if i % 10 == 9 else "normal"
        df, meta = make_flight(f"FL{i:04d}", kind, rng)
        out.append((df, meta))
    return out
# ---------------------------------------------------------------------------

# ---- 전처리·규칙 탐지 공통 함수 ----
SPOOF_THR = {"clear": 12.0, "night": 12.0, "windy": 15.0, "rain": 19.0}   # 기상별 GPS-추측항법 불일치 허용치(m)


def clean(df, weather="clear"):
    """전처리: 중복 제거·정렬 -> 1초 재표본화 -> 결측 보간 -> 고도 스파이크 제거 -> GPS 스푸핑 구간 추측항법 보정."""
    d = df.drop_duplicates("t").sort_values("t").set_index("t")
    d = d.reindex(np.arange(int(d.index.min()), int(d.index.max()) + 1))
    n_missing = int(d["x"].isna().sum())
    num = ["x", "y", "alt", "speed", "rc", "sats", "hdop"]
    d[num] = d[num].interpolate(limit_direction="both")
    hd = d["heading"].dropna()                                   # 방위는 0/360 경계를 풀어서 보간
    un = pd.Series(np.degrees(np.unwrap(np.radians(hd.to_numpy(float)))), index=hd.index)
    d["heading"] = un.reindex(d.index).interpolate(limit_direction="both").to_numpy() % 360
    med = d["alt"].rolling(5, center=True, min_periods=1).median()
    spike = (d["alt"] - med).abs() > 25
    d.loc[spike, "alt"] = med[spike]
    x, y = d["x"].to_numpy(float), d["y"].to_numpy(float)
    r = np.radians(d["heading"].to_numpy(float)); v = d["speed"].to_numpy(float)
    dr = np.c_[v * np.sin(r), v * np.cos(r)]                  # 보고 속도·방위로 계산한 1초 이동량(추측항법)
    xc, yc, flag = x.copy(), y.copy(), np.zeros(len(x), bool)
    thr = SPOOF_THR.get(weather, 12.0)
    ok_run = 0
    for i in range(1, len(x)):
        px, py = xc[i - 1] + dr[i, 0], yc[i - 1] + dr[i, 1]          # 추측항법 예상 위치
        step_ok = np.hypot(x[i] - x[i - 1] - dr[i, 0], y[i] - y[i - 1] - dr[i, 1]) <= thr
        ok_run = ok_run + 1 if step_ok else 0
        if flag[i - 1]:
            # 스푸핑 구간: GPS가 예상 위치로 돌아오거나, GPS 이동량이 5초 연속 물리적으로 맞으면 GPS로 복귀
            if np.hypot(x[i] - px, y[i] - py) < thr * 2 or ok_run >= 5:
                xc[i], yc[i] = x[i], y[i]
            else:
                xc[i], yc[i], flag[i] = px, py, True
        elif not step_ok:
            xc[i], yc[i], flag[i] = px, py, True                     # GPS 이동량이 물리적으로 맞지 않음 -> 스푸핑 의심
        else:
            xc[i], yc[i] = xc[i - 1] + x[i] - x[i - 1], yc[i - 1] + y[i] - y[i - 1]
    d["x"], d["y"], d["spoof_flag"] = xc, yc, flag
    d = d.reset_index()
    d.attrs.update({"missing": n_missing, "spikes": int(spike.sum()), "spoof_pts": int(flag.sum())})
    return d


def xtrack(px, py, plan):
    """계획 경로(웨이포인트 폐곡선) 대비 이탈 거리(m)."""
    P = np.asarray(plan, float)
    segs = [(P[i], P[(i + 1) % len(P)]) for i in range(len(P) if len(P) > 2 else 1)]
    best = np.full(len(px), np.inf)
    for a, b in segs:
        ab = b - a
        u = np.clip(((px - a[0]) * ab[0] + (py - a[1]) * ab[1]) / max(ab @ ab, 1e-9), 0, 1)
        best = np.minimum(best, np.hypot(px - a[0] - u * ab[0], py - a[1] - u * ab[1]))
    return best


RULES = {"turn_rate": 45.0, "hover_speed": 1.5, "hover_sec": 20, "dive_m": 30.0, "xtrack_m": 250.0}


def rule_flags(d, plan, R=RULES):
    """규칙 기반 이상 비행 판정(기록 단위). 급선회·호버링·급강하·경로 이탈·비인가 구역 진입."""
    hr = np.abs(_wrap(np.diff(d["heading"].to_numpy(float), prepend=d["heading"].iloc[0])))
    slow = (d["speed"] < R["hover_speed"]).to_numpy()
    run = np.zeros(len(slow), int)
    for i in range(len(slow)):
        run[i] = run[i - 1] + 1 if slow[i] and i else int(slow[i])
    hover = np.zeros(len(slow), bool)
    for i in np.where(run >= R["hover_sec"])[0]:
        hover[i - R["hover_sec"] + 1:i + 1] = True
    alt = d["alt"].to_numpy(float)
    drop = alt - np.r_[alt[5:], np.full(5, alt[-1])]
    return {"급선회": hr > R["turn_rate"], "호버링": hover, "급강하": drop > R["dive_m"],
            "경로이탈": xtrack(d["x"].to_numpy(float), d["y"].to_numpy(float), plan) > R["xtrack_m"],
            "비인가구역": np.hypot(d["x"] - NFZ["x"], d["y"] - NFZ["y"]).to_numpy() < NFZ["r"]}

# ---- 구간(30초 창) 특징 + 비지도 이상 탐지 공통 함수 ----
WFEAT = ["alt_mean", "alt_min", "alt_std", "vz_min", "spd_mean", "spd_std", "spd_max", "turn_sum", "turn_max",
         "straight", "rc_mean", "rc_zero", "nfz_min", "nfz_close", "xtrack_max"]


def window_feats(d, plan, win=30, step=10):
    """30초 창을 10초씩 옮기며 고도·속도·선회·직진성·조종신호·비인가 구역 접근·경로 이탈 특징을 계산한다."""
    x, y, alt = d["x"].to_numpy(float), d["y"].to_numpy(float), d["alt"].to_numpy(float)
    spd, rc = d["speed"].to_numpy(float), d["rc"].to_numpy(float)
    hr = np.abs(_wrap(np.diff(d["heading"].to_numpy(float), prepend=d["heading"].iloc[0])))
    vz = np.diff(alt, prepend=alt[0])
    nd = np.hypot(x - NFZ["x"], y - NFZ["y"])
    xt = xtrack(x, y, plan)
    step_len = np.hypot(np.diff(x, prepend=x[0]), np.diff(y, prepend=y[0]))
    rows, starts = [], []
    for s in range(0, len(d) - win + 1, step):
        e = s + win
        path = step_len[s + 1:e].sum()
        rows.append([alt[s:e].mean(), alt[s:e].min(), alt[s:e].std(), vz[s:e].min(), spd[s:e].mean(), spd[s:e].std(),
                     spd[s:e].max(), hr[s:e].sum(), hr[s:e].max(), np.hypot(x[e - 1] - x[s], y[e - 1] - y[s]) / max(path, 1.0),
                     rc[s:e].mean(), (rc[s:e] < 5).mean(), nd[s:e].min(), (nd[s] - nd[e - 1]) / win, xt[s:e].max()])
        starts.append(int(d["t"].iloc[s]))
    return np.array(rows), starts


class FlightAnomalyModel:
    """평시 비행로그(라벨 없음)로 통상 패턴 군집(KMeans)과 고립 숲(IsolationForest)을 학습하고,
    두 점수를 학습 구간 분위수로 나눠 1.0을 넘으면 '통상 패턴에서 벗어남'으로 본다."""

    def __init__(self, q=0.999, k=4):
        self.q, self.k = q, k

    def fit(self, X):
        self.sc = StandardScaler().fit(X)
        Z = self.sc.transform(X)
        self.iso = IsolationForest(n_estimators=150, random_state=0).fit(Z)
        self.km = KMeans(self.k, n_init=5, random_state=0).fit(Z)
        self.set_q(self.q, X)
        return self

    def _raw(self, X):
        Z = self.sc.transform(X)
        return -self.iso.score_samples(Z), self.km.transform(Z).min(axis=1)

    def set_q(self, q, X):
        a, b = self._raw(X)
        self.q, self.t_if, self.t_km = q, np.quantile(a, q), np.quantile(b, q)

    def score(self, X):
        a, b = self._raw(X)
        return np.maximum(a / self.t_if, b / self.t_km)


AFEAT = ["ml_score", "low_alt", "rc_weak", "nfz_min_km", "hover_s", "max_speed", "night",
         "pf_commercial", "pf_military", "pf_unknown", "fw_stock", "fw_custom", "fw_unknown"]


def attr_features(d, m, ml_score):
    """운용 주체 추정용 비행 단위 특징. 기체·펌웨어 정보는 상용 기체가 모든 주체에 쓰이므로 약한 단서일 뿐이다."""
    return [min(ml_score, 10.0), float((d["alt"] < 60).mean()), float((d["rc"] < 25).mean()),
            float(np.hypot(d["x"] - NFZ["x"], d["y"] - NFZ["y"]).min() / 1000), float((d["speed"] < 1.5).sum()),
            float(d["speed"].max()), float(m["weather"] == "night"),
            *[float(m["platform"] == p) for p in ("commercial", "military", "unknown")],
            *[float(m["firmware"] == f) for f in ("stock", "custom", "unknown")]]


def prepare(data):
    out = []
    for df, m in data:
        d = clean(df, m["weather"])
        X, st = window_feats(d, m["plan"])
        out.append({"m": m, "d": d, "X": X, "starts": st, "rule": any(v.any() for v in rule_flags(d, m["plan"]).values())})
    return out


# 1) 학습: 라벨 없는 평시 비행로그(순찰·측량) 120건의 30초 창
train = prepare(make_dataset(120, seed=21, anomaly_ratio=0.0))
Xtr = np.vstack([f["X"] for f in train])
model = FlightAnomalyModel(q=0.999).fit(Xtr)
print(f"학습 창 {len(Xtr)}개, 통상 패턴 군집 {model.k}개 크기:", np.bincount(model.km.labels_).tolist())

# 2) 시험: 정상 100건 + 복합 이상 10종 x 10건
test = prepare(make_dataset(200, seed=11))
t0 = time.perf_counter()
for f in test:
    f["ml_score"] = float(model.score(f["X"]).max())
lat = (time.perf_counter() - t0) / sum(len(f["X"]) for f in test) * 1000
for f in test:
    f["ml"] = f["ml_score"] > 1.0
    f["hybrid"] = f["ml"] or f["rule"]
r = pd.DataFrame([{"kind": f["m"]["kind"], "truth": f["m"]["kind"] in SCENARIOS, "규칙": f["rule"], "ML": f["ml"], "규칙+ML": f["hybrid"]} for f in test])
an, nm = r[r.truth], r[~r.truth]
print("\n방식      이상 재현율  정상 비행 오경보율")
for c in ["규칙", "ML", "규칙+ML"]:
    print(f"{c:7s}  {an[c].mean():10.1%}  {nm[c].mean():14.1%}")
rec = an["규칙+ML"].mean()
print(f"\n복합 위협 탐지 재현율 {rec:.1%} (목표 85% 이상: {'달성' if rec >= 0.85 else '미달'}), 30초 창당 처리 {lat:.2f} ms (목표 200 ms 이하)")

# 3) 복합 위협 시나리오 10종 탐지 시험(시나리오별 재현율 80% 이상이면 통과)
per = an.groupby("kind")[["규칙", "ML", "규칙+ML"]].mean()
per["판정"] = np.where(per["규칙+ML"] >= 0.8, "통과", "미통과")
print("\n[시나리오별 재현율]")
print(per.round(2).to_string())
print(f"통과 {int((per['판정'] == '통과').sum())}/10종")

# 4) 오경보 유발 패턴: 정상인데 ML이 이상으로 본 비행의 가장 튀는 특징
fa = [f for f in test if f["m"]["kind"] not in SCENARIOS and f["ml"]]
print(f"\n[정상 비행 오경보 {len(fa)}건의 원인 특징]")
for f in fa[:5]:
    w = int(np.argmax(model.score(f["X"])))
    z = model.sc.transform(f["X"][w:w + 1])[0]
    top = np.argsort(-np.abs(z))[:2]
    print(f" {f['m']['id']} ({f['m']['kind']}, {f['m']['weather']}): " + ", ".join(f"{WFEAT[i]} z={z[i]:+.1f}" for i in top))
print("-> 강풍 돌풍 하강·측량 U턴처럼 정상이지만 드문 패턴이 원인이면 학습 자료에 추가하거나 특징을 재설계한다.")

# 5) 그림: 이상 점수 분포와 은밀 침투 예시
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].hist([f["ml_score"] for f in test if f["m"]["kind"] not in SCENARIOS], bins=30, range=(0, 6), alpha=.7, label="normal")
ax[0].hist([f["ml_score"] for f in test if f["m"]["kind"] in SCENARIOS], bins=30, range=(0, 6), alpha=.7, label="anomalous")
ax[0].axvline(1.0, color="k", ls="--"); ax[0].set_xlabel("max window anomaly score"); ax[0].set_ylabel("flights"); ax[0].legend()
ax[0].set_title("Anomaly score by flight type")
f = next(f for f in test if f["m"]["kind"] == "infiltration")
s = model.score(f["X"])
ax[1].plot(f["starts"], s, "-o", ms=3, label="window score"); ax[1].axhline(1.0, color="k", ls="--")
ax[1].plot(f["d"]["t"], f["d"]["alt"] / 30, color="tab:gray", label="altitude / 30 m")
ax[1].set_xlabel("time (s)"); ax[1].set_title(f"{f['m']['id']} infiltration: low altitude + RC silent"); ax[1].legend()
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
학습 창 3360개, 통상 패턴 군집 4개 크기: [1104, 76, 1316, 864]

방식      이상 재현율  정상 비행 오경보율
규칙            57.0%           13.0%
ML           100.0%            8.0%
규칙+ML        100.0%           16.0%

복합 위협 탐지 재현율 100.0% (목표 85% 이상: 달성), 30초 창당 처리 0.47 ms (목표 200 ms 이하)

[시나리오별 재현율]
               규칙   ML  규칙+ML  판정
kind                             
deviation     1.0  1.0    1.0  통과
dive          1.0  1.0    1.0  통과
hover         1.0  1.0    1.0  통과
infiltration  0.0  1.0    1.0  통과
loiter        0.3  1.0    1.0  통과
low_level     0.2  1.0    1.0  통과
nfz_entry     1.0  1.0    1.0  통과
rc_silent     0.1  1.0    1.0  통과
sharp_turn    1.0  1.0    1.0  통과
speed_burst   0.1  1.0    1.0  통과
통과 10/10종

[정상 비행 오경보 8건의 원인 특징]
 FL0019 (survey, windy): alt_std z=+6.7, vz_min z=-6.7
 FL0039 (survey, windy): alt_std z=+8.9, vz_min z=-8.7
 FL0059 (survey, clear): straight z=-5.5, turn_max z=+4.1
 FL0088 (normal, windy): alt_std z=+7.6, alt_min z=-2.6
 FL0089 (survey, rain): straight z=-5.6, turn_max z=+4.1
-> 강풍 돌풍 하강·측량 U턴처럼 정상이지만 드문 패턴이 원인이면 학습 자료에 추가하거나 특징을 재설계한다.
```
</details>
